# Original-pipeline walkthrough

A guided, runnable tour of the empirical experiments behind *"Benchmark Scores
Are Pipeline-Dependent: A Reliability Audit of Cybersecurity LLM Benchmarks."*

Each notebook calls the existing analysis scripts and renders pre-computed
outputs. No logic is duplicated here — the scripts are the source of truth.

## How to use
- Run cells top-to-bottom. Light steps (loading results, plotting) recompute
  instantly from `outputs_final/`. If the central storage isn't mounted they
  fall back gracefully with a note.
- **Heavy steps are off by default.** GPU inference and API calls are pre-computed;
  stored outputs load instantly.

## The measurement-pipeline framing

Every benchmark is modelled as a 5-stage pipeline:

```
Dataset 𝒟  ·  Prompt 𝒫  ·  Inference 𝓘  ·  Extraction/scoring 𝓔  ·  Aggregation 𝒜
```

A reported score is conditional on *all five stages*, not an intrinsic model
property. The experiments below hold 𝒟 fixed and vary each other stage.

## Notebook map
| Notebook | Theme | Paper section |
|---|---|---|
| `01_results_table` | Full 10-model × 25-task results (paper Table 2) + LaTeX table | §5 |
| `02_failure_modes` | 15 failure-mode headline numbers | §4 |
| `03_prompt_sensitivity` | Zero-shot vs few-shot vs CoT across 10 models (paper Table 2) | §4.4 F4(E) |
| `04_inference_config` | Stop-sequence, token-budget, temperature | §4.3 F1–3(I) |
| `05_logprob_vs_generative` | RedSage logprob vs generative; TAA drift | §4.5 F1–3(A) |

## How results were collected

**Inference** — `run_inference_benchmarks.py` collected model responses for all 10 models
across 8 benchmarks (23 tasks). Open-weight models were served locally via vLLM;
proprietary models (GPT-5.4, Claude Sonnet 4.6) were queried through their APIs.

**Evaluation** — `evaluate.py` applied benchmark-faithful scoring to collected responses
(task-specific extractors, exact match, CWE regex, CVSS MAD, etc.), producing
`{task}_result.json` and `{task}_detail.jsonl` per model.
`run_evaluate_llm_judge.py` ran the unified LLM-as-judge extraction for open-ended tasks
(ATE, TAA, RCM) where regex-based extraction is insufficient.

**Prompt sensitivity** — `run_prompt_sensitivity.py` re-ran 100 seeded samples per task
under zero-shot, 2-shot few-shot, and chain-of-thought prompting for all 10 models.

**RedSage logprob** — `run_redsage_lighteval.py` ran the official LightEval logprob
and generative exact-match evaluation on RedSage-Bench.

All outputs are pre-computed and stored in the repository — notebooks load directly
from stored files, no inference or API calls required.


In [1]:
import nbtools as nb
from nbtools import (
    show_df, show_fig, show_md, show_tex,
    run_mod, run_live, heavy,
    REGEN, OUTPUTS_FINAL, OUTPUTS_DIR, PIPELINE_ROOT,
)
import pandas as pd
from IPython.display import display, HTML
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.1f}'.format)

# The 10 models evaluated in the paper (Table 2).
PAPER_MODELS = {
    'Claude Sonnet 4.6':   'claude-sonnet-4-6-cyberxpert',
    'GPT-5.4':             'gpt-5.4',
    'Gemma-4-31B':         'gemma-4-31B-it',
    'Qwen3.6-35B':         'Qwen3.6-35B-A3B',
    'Llama-3.3-70B':       'Llama-3.3-70B-Instruct',
    'GPT-OSS-20B':         'gpt-oss-20b',
    'Primus-Nemotron-70B': 'Llama-Primus-Nemotron-70B',
    'Primus-Merged-8B':    'Llama-Primus-Merged',
    'Foundation-Sec-8B':   'Foundation-Sec-8B-Instruct',
    'RedSage-Qwen3-8B':    'RedSage-Qwen3-8B-DPO',
}

print(f"Data root: {OUTPUTS_FINAL.name}/  — {'OK' if OUTPUTS_FINAL.exists() else 'MISSING'}")

Data root: outputs_final/  — OK


### Evaluated models

In [2]:
PAPER_DIR_NAMES = set(PAPER_MODELS.values())
if OUTPUTS_FINAL.exists():
    rows = []
    for label, dir_name in PAPER_MODELS.items():
        p = OUTPUTS_FINAL / dir_name
        n_tasks = len(list((p / 'eval').glob('*_result.json'))) if (p / 'eval').exists() else 0
        rows.append({'Model': label, 'Tasks evaluated': n_tasks})
    inv = pd.DataFrame(rows)
    display(inv.style
            .hide(axis='index')
            .set_caption('Models evaluated in the paper — task count from outputs_final/.'))
else:
    print("[missing] Result files not found.")

Model,Tasks evaluated
Claude Sonnet 4.6,24
GPT-5.4,24
Gemma-4-31B,25
Qwen3.6-35B,25
Llama-3.3-70B,25
GPT-OSS-20B,25
Primus-Nemotron-70B,25
Primus-Merged-8B,25
Foundation-Sec-8B,25
RedSage-Qwen3-8B,25
